In [111]:
# Code adapted from:
# 1. https://huggingface.co/docs/transformers/en/training
# 2. https://huggingface.co/docs/transformers/en/tasks/sequence_classification

In [151]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer
from torch.nn import LayerNorm
from torch.optim import AdamW, lr_scheduler
from transformers.trainer_pt_utils import get_parameter_names
from datasets import load_dataset
from pathlib import Path
import numpy as np
import evaluate

In [134]:
dataset = load_dataset("json", data_files={"train": ["./out/ot_train.json", "./out/nt_train.json"], "validation": ["./out/ot_test.json", "./out/nt_test.json"]})

In [136]:
print(dataset["train"][0])

{'label': 0, 'text': 'BRCJT BR> >LH> JT CMJ> WJT >R<>'}


In [137]:
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilroberta-base")

In [138]:
def tokenize_function(data):
    return tokenizer(data["text"], padding="max_length", truncation=True)

In [139]:
tokenized_data = dataset.map(
    tokenize_function,
    batched=True,
    num_proc=4,
)

Map (num_proc=4):   0%|          | 0/5841 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/1562 [00:00<?, ? examples/s]

In [140]:
print(tokenized_data)

DatasetDict({
    train: Dataset({
        features: ['label', 'text', 'input_ids', 'attention_mask'],
        num_rows: 5841
    })
    validation: Dataset({
        features: ['label', 'text', 'input_ids', 'attention_mask'],
        num_rows: 1562
    })
})


In [141]:
id2label = {0: "Jewish", 1: "Christian"}
label2id = {"Jewish": 0, "Christian": 1}

In [142]:
model = AutoModelForSequenceClassification.from_pretrained("distilbert/distilbert-base-cased", num_labels=2, id2label=id2label, label2id=label2id, torch_dtype="auto")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [143]:
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [144]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [145]:
training_args = TrainingArguments(
    output_dir="peshitta_trainer",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    eval_strategy="epoch",
    do_eval=True,
    # seed=SEED
)

In [146]:
# Adapted from: https://github.com/huggingface/transformers/issues/18635#issuecomment-1216860652
decay_parameters = get_parameter_names(model, [LayerNorm])
decay_parameters = [name for name in decay_parameters if "bias" not in name]

optimizer_grouped_parameters = [
        {
            "params": [p for n, p in model.named_parameters() if n in decay_parameters],
            "weight_decay": 0.01,
        },
        {
            "params": [p for n, p in model.named_parameters() if n not in decay_parameters],
            "weight_decay": 0.0,
        },
    ]

optimizer = AdamW(
    optimizer_grouped_parameters,
    lr=2e-5,
    eps=1e-8
)

# params, lr=0.001, betas=(0.9, 0.999), eps=1e-08, weight_decay=0.01, amsgrad=False, *, maximize=False, foreach=None, capturable=False, differentiable=False, fused=None

In [147]:
scheduler = lr_scheduler.LinearLR(optimizer)

In [148]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, scheduler)
)

In [149]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.391326,0.884763
2,No log,0.345234,0.888604
3,0.164200,0.313563,0.886684


TrainOutput(global_step=549, training_loss=0.16013083866168024, metrics={'train_runtime': 1957.2646, 'train_samples_per_second': 8.953, 'train_steps_per_second': 0.28, 'total_flos': 2321226226649088.0, 'train_loss': 0.16013083866168024, 'epoch': 3.0})

In [150]:
trainer.save_model(str(Path("./aabert").absolute()))

In [166]:
tokenized_data["validation"][0]["text"]

'WHLJN PT"GM> D>MR MWC> LKLH >JSRJL B<BR> DJWRDNN BMDBR> B<RB> LWQBL SWP BJT PRN WBJT TPL WLBNN WXYRWT WRZHB'

In [171]:
inputs = tokenizer(dataset["train"][0]["text"], padding="max_length", truncation=True, return_tensors="pt")
outputs = model(**inputs)

RuntimeError: Placeholder storage has not been allocated on MPS device!

In [159]:
# sentencepiece

In [ ]:
# serious re-implementation for real data